In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import uuid

CATALOG = "patient_kg_dev"
STAGED = f"{CATALOG}.staged"
SILVER = f"{CATALOG}.silver"
QUALITY = f"{CATALOG}.quality"

QUALITY_RUN_ID = str(uuid.uuid4())
QUALITY_RUN_AT = datetime.utcnow()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {QUALITY}")

NORMALIZATION_VERSION = "v1"
print("Quality run:", QUALITY_RUN_ID)

In [0]:
quality_results = []

def record_check(
    rule_id,
    dataset,
    severity,
    violation_count,
    description,
    allowed_violations=0
):
    violation_count = int(violation_count)

    status = (
        "PASS"
        if violation_count <= allowed_violations
        else "FAIL"
    )

    quality_results.append({
        "quality_run_id": QUALITY_RUN_ID,
        "quality_run_at": QUALITY_RUN_AT,
        "rule_id": rule_id,
        "dataset": dataset,
        "severity": severity,
        "violation_count": violation_count,
        "allowed_violations": allowed_violations,
        "status": status,
        "description": description
    })

# Part A: Required identifier checks
**Missing patient identifiers**

In [0]:
patient_based_tables = [
    "patients",
    "encounters",
    "conditions",
    "medications",
    "procedures",
    "observations",
    "careplans"
]

for table_name in patient_based_tables:
    violations = (
        spark.table(f"{SILVER}.{table_name}")
        .filter(
            F.col("patient_id").isNull() |
            (F.trim(F.col("patient_id")) == "")
        )
        .count()
    )

    record_check(
        rule_id="required_patient_id",
        dataset=table_name,
        severity="ERROR",
        violation_count=violations,
        description="Every Silver record must identify its patient."
    )

Unique source identifiers

In [0]:
explicit_ids = {
    "patients": "patient_id",
    "encounters": "encounter_id",
    "careplans": "careplan_id"
}

for table_name, id_column in explicit_ids.items():
    duplicate_count = (
        spark.table(f"{SILVER}.{table_name}")
        .groupBy(id_column)
        .count()
        .filter(
            F.col(id_column).isNotNull() &
            (F.col("count") > 1)
        )
        .select(
            F.coalesce(
                F.sum(F.col("count") - 1),
                F.lit(0)
            ).alias("duplicate_rows")
        )
        .first()["duplicate_rows"]
    )

    record_check(
        rule_id=f"unique_{id_column}",
        dataset=table_name,
        severity="ERROR",
        violation_count=duplicate_count,
        description=f"{id_column} must be unique within {table_name}."
    )

# Part B: Relationship integrity
**Patient relationship checks**

In [0]:
patient_ids = (
    spark.table(f"{SILVER}.patients")
    .select("patient_id")
    .distinct()
)

for table_name in [
    "encounters",
    "conditions",
    "medications",
    "procedures",
    "observations",
    "careplans"
]:
    orphan_count = (
        spark.table(f"{SILVER}.{table_name}")
        .select("patient_id")
        .join(patient_ids, "patient_id", "left_anti")
        .count()
    )

    record_check(
        rule_id="patient_relationship_integrity",
        dataset=table_name,
        severity="ERROR",
        violation_count=orphan_count,
        description=(
            "Every patient reference must resolve to a patient "
            "in the Silver cohort."
        )
    )

**Encounter relationship checks:**

In [0]:
encounter_ids = (
    spark.table(f"{SILVER}.encounters")
    .select("encounter_id")
    .distinct()
)

for table_name in [
    "conditions",
    "medications",
    "procedures",
    "observations",
    "careplans"
]:
    orphan_count = (
        spark.table(f"{SILVER}.{table_name}")
        .filter(F.col("encounter_id").isNotNull())
        .select("encounter_id")
        .join(encounter_ids, "encounter_id", "left_anti")
        .count()
    )

    record_check(
        rule_id="encounter_relationship_integrity",
        dataset=table_name,
        severity="ERROR",
        violation_count=orphan_count,
        description=(
            "Every non-null encounter reference must resolve "
            "to a Silver encounter."
        )
    )

# Part C: Date and timestamp validity

**Invalid date intervals:**

In [0]:
temporal_rules = [
    ("encounters", "start_at", "stop_at"),
    ("conditions", "start_date", "stop_date"),
    ("medications", "start_at", "stop_at"),
    ("procedures", "start_at", "stop_at"),
    ("careplans", "start_date", "stop_date")
]

for table_name, start_column, stop_column in temporal_rules:
    invalid_intervals = (
        spark.table(f"{SILVER}.{table_name}")
        .filter(
            F.col(start_column).isNotNull() &
            F.col(stop_column).isNotNull() &
            (F.col(start_column) > F.col(stop_column))
        )
        .count()
    )

    record_check(
        rule_id="valid_temporal_interval",
        dataset=table_name,
        severity="ERROR",
        violation_count=invalid_intervals,
        description=(
            f"{start_column} cannot occur after {stop_column}."
        )
    )

**Timestamp parsing checks**

In [0]:
parsing_rules = [
    ("patients", "birth_date_source", "birth_date"),
    ("encounters", "start_at_source", "start_at"),
    ("encounters", "stop_at_source", "stop_at"),
    ("conditions", "start_date_source", "start_date"),
    ("medications", "start_at_source", "start_at"),
    ("procedures", "start_at_source", "start_at"),
    ("observations", "observed_at_source", "observed_at"),
    ("careplans", "start_date_source", "start_date")
]

for table_name, source_column, parsed_column in parsing_rules:
    parse_failures = (
        spark.table(f"{SILVER}.{table_name}")
        .filter(
            F.col(source_column).isNotNull() &
            (F.trim(F.col(source_column)) != "") &
            F.col(parsed_column).isNull()
        )
        .count()
    )

    record_check(
        rule_id=f"parse_{parsed_column}",
        dataset=table_name,
        severity="ERROR",
        violation_count=parse_failures,
        description=(
            f"Nonblank {source_column} values must successfully "
            f"parse into {parsed_column}."
        )
    )

# Part D: Known observation nuances

In [0]:
encounterless_observations = (
    spark.table(f"{SILVER}.observations")
    .filter(F.col("encounter_id").isNull())
    .withColumn("quality_run_id", F.lit(QUALITY_RUN_ID))
    .withColumn("quality_reason", F.lit(
        "Observation has no source encounter association"
    ))
)

encounterless_observations.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{QUALITY}.encounterless_observations")

encounterless_count = encounterless_observations.count()

record_check(
    rule_id="encounterless_observations",
    dataset="observations",
    severity="INFO",
    violation_count=encounterless_count,
    allowed_violations=encounterless_count,
    description=(
        "Encounter association is optional for some observations. "
        "These rows are retained."
    )
)

In [0]:
observation_duplicate_groups = (
    spark.table(f"{SILVER}.observations")
    .groupBy("_row_content_sha256")
    .agg(
        F.count("*").alias("row_count"),
        F.first("patient_id").alias("patient_id"),
        F.first("encounter_id").alias("encounter_id"),
        F.first("observed_at").alias("observed_at"),
        F.first("code").alias("code"),
        F.first("description_source").alias("description_source"),
        F.first("value_source").alias("value_source"),
        F.first("unit_source").alias("unit_source")
    )
    .filter(F.col("row_count") > 1)
    .withColumn(
        "duplicate_rows_beyond_first",
        F.col("row_count") - 1
    )
    .withColumn("quality_run_id", F.lit(QUALITY_RUN_ID))
)

observation_duplicate_groups.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{QUALITY}.observation_duplicate_groups")

In [0]:
duplicate_observation_count = (
    observation_duplicate_groups
    .select(
        F.coalesce(
            F.sum("duplicate_rows_beyond_first"),
            F.lit(0)
        ).alias("duplicate_count")
    )
    .first()["duplicate_count"]
)

record_check(
    rule_id="duplicate_source_rows",
    dataset="observations",
    severity="WARNING",
    violation_count=duplicate_observation_count,
    description=(
        "Exact duplicate observation rows were detected. "
        "Silver preserves them; Graph-ready will apply a documented "
        "deduplication rule."
    )
)

In [0]:
display(
    spark.table(f"{SILVER}.observations")
    .groupBy("value_type_source")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
numeric_observation_parse_failures = (
    spark.table(f"{SILVER}.observations")
    .filter(
        (F.lower(F.col("value_type_source")) == "numeric") &
        F.col("value_text").isNotNull() &
        F.col("value_numeric").isNull()
    )
    .count()
)

record_check(
    rule_id="numeric_observation_value_parse",
    dataset="observations",
    severity="ERROR",
    violation_count=numeric_observation_parse_failures,
    description=(
        "Observations identified as numeric must have a usable "
        "numeric representation."
    )
)

# Lineage Completeness

In [0]:
for table_name in [
    "patients",
    "encounters",
    "conditions",
    "medications",
    "procedures",
    "observations",
    "careplans"
]:
    missing_lineage = (
        spark.table(f"{SILVER}.{table_name}")
        .filter(
            F.col("_source_file").isNull() |
            F.col("_source_file_path").isNull() |
            F.col("_row_content_sha256").isNull() |
            F.col("_ingestion_run_id").isNull() |
            F.col("_ingested_at").isNull()
        )
        .count()
    )

    record_check(
        rule_id="lineage_completeness",
        dataset=table_name,
        severity="ERROR",
        violation_count=missing_lineage,
        description=(
            "Every Silver row must retain its Bronze ingestion lineage."
        )
    )

In [0]:
required_lineage_columns = [
    "_source_file",
    "_source_file_path",
    "_row_content_sha256",
    "_ingestion_run_id",
    "_ingested_at"
]

for table_name in [
    "patients",
    "encounters",
    "conditions",
    "medications",
    "procedures",
    "observations",
    "careplans"
]:
    actual_columns = set(
        spark.table(f"{SILVER}.{table_name}").columns
    )

    missing_columns = [
        column_name
        for column_name in required_lineage_columns
        if column_name not in actual_columns
    ]

    print(table_name, "missing:", missing_columns)

In [0]:
def blank_to_null(column_name):
    return F.when(
        F.trim(F.col(column_name)) == "",
        None
    ).otherwise(F.trim(F.col(column_name)))

print("Staged:", STAGED)
print("Silver:", SILVER)
print("Quality:", QUALITY)

In [0]:
print(
    spark.table(f"{SILVER}.patients")
    .select(
        "_ingestion_run_id",
        "_ingested_at"
    )
    .count()
)

In [0]:
patients = (
    spark.table(f"{STAGED}.patients")
    .select(
        blank_to_null("Id").alias("patient_id"),

        F.col("BIRTHDATE").alias("birth_date_source"),
        F.to_date("BIRTHDATE", "yyyy-MM-dd").alias("birth_date"),

        F.col("DEATHDATE").alias("death_date_source"),
        F.to_date("DEATHDATE", "yyyy-MM-dd").alias("death_date"),

        blank_to_null("GENDER").alias("gender"),
        blank_to_null("RACE").alias("race"),
        blank_to_null("ETHNICITY").alias("ethnicity"),
        blank_to_null("CITY").alias("city"),
        blank_to_null("STATE").alias("state"),

        F.col("HEALTHCARE_EXPENSES").alias(
            "healthcare_expenses_source"
        ),
        F.expr(
            "try_cast(HEALTHCARE_EXPENSES AS DECIMAL(18,2))"
        ).alias("healthcare_expenses"),

        F.col("HEALTHCARE_COVERAGE").alias(
            "healthcare_coverage_source"
        ),
        F.expr(
            "try_cast(HEALTHCARE_COVERAGE AS DECIMAL(18,2))"
        ).alias("healthcare_coverage"),

        F.col("INCOME").alias("income_source"),
        F.expr(
            "try_cast(INCOME AS DECIMAL(18,2))"
        ).alias("income"),

        F.col("_source_file"),
        F.col("_source_file_path"),
        F.col("_source_file_size"),
        F.col("_source_file_modified_at"),
        F.col("_row_content_sha256"),

        # These were missing previously
        F.col("_ingestion_run_id"),
        F.col("_ingested_at"),

        F.lit(NORMALIZATION_VERSION).alias(
            "_normalization_version"
        ),
        F.current_timestamp().alias("_normalized_at")
    )
)

patients.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.patients")

In [0]:
quality_results_df = spark.createDataFrame(quality_results)

quality_results_df.write.mode("append").saveAsTable(
    f"{QUALITY}.check_results"
)

display(
    quality_results_df.orderBy(
        "severity",
        "dataset",
        "rule_id"
    )
)

In [0]:
additional_parsing_rules = [
    ("patients", "death_date_source", "death_date"),
    ("conditions", "stop_date_source", "stop_date"),
    ("medications", "stop_at_source", "stop_at"),
    ("procedures", "stop_at_source", "stop_at"),
    ("careplans", "stop_date_source", "stop_date")
]

for table_name, source_column, parsed_column in additional_parsing_rules:
    parse_failures = (
        spark.table(f"{SILVER}.{table_name}")
        .filter(
            F.col(source_column).isNotNull() &
            (F.trim(F.col(source_column)) != "") &
            F.col(parsed_column).isNull()
        )
        .count()
    )

    record_check(
        rule_id=f"parse_{parsed_column}",
        dataset=table_name,
        severity="ERROR",
        violation_count=parse_failures,
        description=(
            f"Nonblank {source_column} values must successfully "
            f"parse into {parsed_column}."
        )
    )

In [0]:
quality_results_df = spark.createDataFrame(quality_results)

display(
    quality_results_df.orderBy(
        "severity",
        "dataset",
        "rule_id"
    )
)

In [0]:
display(
    quality_results_df.filter(
        F.col("rule_id").isin(
            "lineage_schema_completeness",
            "lineage_value_completeness"
        )
    )
)

In [0]:
blocking_failures = quality_results_df.filter(
    (F.col("severity") == "ERROR") &
    (F.col("status") == "FAIL")
)

display(blocking_failures)

assert blocking_failures.count() == 0, \
    "Quality gate has blocking failures"

print("QUALITY GATE PASSED")

In [0]:
quality_results_df.write.mode("append").saveAsTable(
    f"{QUALITY}.check_results"
)

In [0]:
required_lineage_columns = [
    "_source_file",
    "_source_file_path",
    "_row_content_sha256",
    "_ingestion_run_id",
    "_ingested_at"
]

for table_name in [
    "patients",
    "encounters",
    "conditions",
    "medications",
    "procedures",
    "observations",
    "careplans"
]:
    table_df = spark.table(f"{SILVER}.{table_name}")
    actual_columns = set(table_df.columns)

    missing_columns = [
        column_name
        for column_name in required_lineage_columns
        if column_name not in actual_columns
    ]

    record_check(
        rule_id="lineage_schema_completeness",
        dataset=table_name,
        severity="ERROR",
        violation_count=len(missing_columns),
        description=(
            "Missing required lineage columns: "
            + (
                ", ".join(missing_columns)
                if missing_columns
                else "none"
            )
        )
    )

    if not missing_columns:
        missing_values = (
            table_df
            .filter(
                F.col("_source_file").isNull() |
                F.col("_source_file_path").isNull() |
                F.col("_row_content_sha256").isNull() |
                F.col("_ingestion_run_id").isNull() |
                F.col("_ingested_at").isNull()
            )
            .count()
        )

        record_check(
            rule_id="lineage_value_completeness",
            dataset=table_name,
            severity="ERROR",
            violation_count=missing_values,
            description=(
                "Every Silver record must contain complete "
                "ingestion lineage."
            )
        )

In [0]:
quality_results_df = spark.createDataFrame(quality_results)

In [0]:
lineage_results = quality_results_df.filter(
    F.col("rule_id").isin(
        "lineage_schema_completeness",
        "lineage_value_completeness"
    )
)

display(lineage_results.orderBy("dataset", "rule_id"))

print("Lineage check rows:", lineage_results.count())

In [0]:
blocking_failures = quality_results_df.filter(
    (F.col("severity") == "ERROR") &
    (F.col("status") == "FAIL")
)

display(blocking_failures)

assert blocking_failures.count() == 0, \
    "Quality gate has blocking failures"

print("QUALITY GATE PASSED")

In [0]:
quality_results_df.write.mode("append").saveAsTable(
    f"{QUALITY}.check_results"
)